In [1]:
import $ivy.`org.apache.spark::spark-sql:3.5.8`
import $ivy.`org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.10.1`
import $ivy.`org.apache.iceberg:iceberg-aws-bundle:1.10.1`

import $ivy.$
import $ivy.$
import $ivy.$

In [2]:
import org.apache.spark.sql._
import org.apache.spark.sql.types._
import org.apache.spark.sql.functions._

import org.apache.spark.sql._
import org.apache.spark.sql.types._
import org.apache.spark.sql.functions._

In [3]:
val spark = SparkSession
                .builder()
                .master("local[*]")
                .appName("Iceberg")
                .config("spark.log.level", "WARN")
                .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
                .config("spark.sql.defaultCatalog", "lakekeeper")
                .config("spark.sql.catalog.lakekeeper", "org.apache.iceberg.spark.SparkCatalog")
                .config("spark.sql.catalog.lakekeeper.type", "rest")
                .config("spark.sql.catalog.lakekeeper.uri", "http://lakekeeper:9191/catalog")
                .config("spark.sql.catalog.lakekeeper.warehouse", "demo")
                .config("spark.sql.catalog.lakekeeper.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
                .getOrCreate()

import spark.implicits._

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/02 23:34:12 INFO SparkContext: Running Spark version 3.5.8
26/07/02 23:34:12 INFO SparkContext: OS info Linux, 6.12.0-211.22.1.el10_2.x86_64, amd64
26/07/02 23:34:12 INFO SparkContext: Java version 11.0.31
26/07/02 23:34:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting Spark log level to "WARN".
26/07/02 23:34:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


spark: SparkSession = org.apache.spark.sql.SparkSession@4b4a0547
import spark.implicits._

In [4]:
println(s"spark.version == ${spark.version}")

spark.version == 3.5.8


# Создаём namespace

In [5]:
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS spark_demo")
spark.sql("SHOW NAMESPACES").show()

+--------------+
|     namespace|
+--------------+
|pyiceberg_demo|
|    spark_demo|
+--------------+



res5_0: DataFrame = []

# Создаём таблицу

In [6]:
spark.sql("DROP TABLE IF EXISTS spark_demo.taxis")

res6: DataFrame = []

In [7]:
val schema = StructType( Array(
    StructField("vendor_id", LongType, true),
    StructField("trip_id", LongType, true),
    StructField("trip_distance", FloatType, true),
    StructField("fare_amount", DoubleType, true),
    StructField("store_and_fwd_flag", StringType, true)
))

schema: StructType = StructType(
  StructField("vendor_id", LongType, true, {}),
  StructField("trip_id", LongType, true, {}),
  StructField("trip_distance", FloatType, true, {}),
  StructField("fare_amount", DoubleType, true, {}),
  StructField("store_and_fwd_flag", StringType, true, {})
)

In [8]:
val df = spark.createDataFrame(spark.sparkContext.emptyRDD[Row], schema)
df.writeTo("spark_demo.taxis").create()

df: DataFrame = [vendor_id: bigint, trip_id: bigint ... 3 more fields]

In [9]:
spark.table("spark_demo.taxis").show()

+---------+-------+-------------+-----------+------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|
+---------+-------+-------------+-----------+------------------+
+---------+-------+-------------+-----------+------------------+



In [10]:
spark.sql("ALTER TABLE spark_demo.taxis CREATE TAG `initial`")

res10: DataFrame = []

# Writing Data to a Table

In [11]:
Thread.sleep(600000)

In [12]:
val data1 = Seq(
    Row(1: Long, 1000371: Long, 1.8f: Float, 15.32: Double, "N": String),
    Row(2: Long, 1000372: Long, 2.5f: Float, 22.15: Double, "N": String),
    Row(2: Long, 1000373: Long, 0.9f: Float, 9.01: Double, "N": String),
    Row(1: Long, 1000374: Long, 8.4f: Float, 42.13: Double, "Y": String)
)

val df1 = spark.createDataFrame(spark.sparkContext.parallelize(data1), schema)

data1: Seq[Row] = List(
  [1,1000371,1.8,15.32,N],
  [2,1000372,2.5,22.15,N],
  [2,1000373,0.9,9.01,N],
  [1,1000374,8.4,42.13,Y]
)
df1: DataFrame = [vendor_id: bigint, trip_id: bigint ... 3 more fields]

In [13]:
df1.writeTo("spark_demo.taxis").append()

In [14]:
spark.table("spark_demo.taxis").show()

+---------+-------+-------------+-----------+------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|
+---------+-------+-------------+-----------+------------------+
|        1|1000371|          1.8|      15.32|                 N|
|        2|1000372|          2.5|      22.15|                 N|
|        2|1000373|          0.9|       9.01|                 N|
|        1|1000374|          8.4|      42.13|                 Y|
+---------+-------+-------------+-----------+------------------+



In [15]:
spark.sql("ALTER TABLE spark_demo.taxis CREATE TAG `first_write`")

res15: DataFrame = []

In [16]:
Thread.sleep(600000)

In [17]:
val data2 = Seq(
    Row(3: Long, 2000371: Long, 1.8f: Float, 15.32: Double, "N": String),
    Row(4: Long, 2000372: Long, 2.5f: Float, 22.15: Double, "N": String),
    Row(4: Long, 2000373: Long, 0.9f: Float, 9.01: Double, "N": String),
    Row(3: Long, 2000374: Long, 8.4f: Float, 42.13: Double, "Y": String)
)

val df2 = spark.createDataFrame(spark.sparkContext.parallelize(data2), schema)
df2.writeTo("spark_demo.taxis").append()

data2: Seq[Row] = List(
  [3,2000371,1.8,15.32,N],
  [4,2000372,2.5,22.15,N],
  [4,2000373,0.9,9.01,N],
  [3,2000374,8.4,42.13,Y]
)
df2: DataFrame = [vendor_id: bigint, trip_id: bigint ... 3 more fields]

In [18]:
spark.table("spark_demo.taxis").show()

+---------+-------+-------------+-----------+------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|
+---------+-------+-------------+-----------+------------------+
|        1|1000371|          1.8|      15.32|                 N|
|        2|1000372|          2.5|      22.15|                 N|
|        2|1000373|          0.9|       9.01|                 N|
|        1|1000374|          8.4|      42.13|                 Y|
|        3|2000371|          1.8|      15.32|                 N|
|        4|2000372|          2.5|      22.15|                 N|
|        4|2000373|          0.9|       9.01|                 N|
|        3|2000374|          8.4|      42.13|                 Y|
+---------+-------+-------------+-----------+------------------+



In [19]:
spark.sql("ALTER TABLE spark_demo.taxis CREATE TAG `second_write`")

res19: DataFrame = []

In [20]:
spark.sql("SELECT * FROM spark_demo.taxis.refs").show()

+------------+------+-------------------+-----------------------+---------------------+----------------------+
|        name|  type|        snapshot_id|max_reference_age_in_ms|min_snapshots_to_keep|max_snapshot_age_in_ms|
+------------+------+-------------------+-----------------------+---------------------+----------------------+
| first_write|   TAG|1040870493480983937|                   NULL|                 NULL|                  NULL|
|        main|BRANCH|5995907169292718909|                   NULL|                 NULL|                  NULL|
|     initial|   TAG|8407173578228161952|                   NULL|                 NULL|                  NULL|
|second_write|   TAG|5995907169292718909|                   NULL|                 NULL|                  NULL|
+------------+------+-------------------+-----------------------+---------------------+----------------------+



# Snapshots

In [21]:
spark.sql("SELECT * FROM spark_demo.taxis.snapshots ORDER BY committed_at").show()

+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|          parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|2026-07-02 23:34:...|8407173578228161952|               NULL|   append|s3://lakehouse/01...|{engine-name -> s...|
|2026-07-02 23:44:...|1040870493480983937|8407173578228161952|   append|s3://lakehouse/01...|{total-records ->...|
|2026-07-02 23:54:...|5995907169292718909|1040870493480983937|   append|s3://lakehouse/01...|{spark.app.id -> ...|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+



In [22]:
spark.sql("""SELECT
    committed_at,
    snapshot_id,
    parent_id
FROM spark_demo.taxis.snapshots
ORDER BY committed_at""")
.show(false)

+-----------------------+-------------------+-------------------+
|committed_at           |snapshot_id        |parent_id          |
+-----------------------+-------------------+-------------------+
|2026-07-02 23:34:20.831|8407173578228161952|NULL               |
|2026-07-02 23:44:23.692|1040870493480983937|8407173578228161952|
|2026-07-02 23:54:26.021|5995907169292718909|1040870493480983937|
+-----------------------+-------------------+-------------------+



In [23]:
val ts = spark.sql("""SELECT
    date_trunc('MINUTE', committed_at + INTERVAL '5 MINUTES'),
    snapshot_id
FROM spark_demo.taxis.snapshots
ORDER BY committed_at""")
    .as[(String, Long)]
    .collect()

ts: Array[(String, Long)] = Array(
  ("2026-07-02 23:39:00", 8407173578228161952L),
  ("2026-07-02 23:49:00", 1040870493480983937L),
  ("2026-07-02 23:59:00", 5995907169292718909L)
)

In [24]:
val ts1 = ts(0)._1
val ts2 = ts(1)._1
val ts3 = ts(2)._1

ts1: String = "2026-07-02 23:39:00"
ts2: String = "2026-07-02 23:49:00"
ts3: String = "2026-07-02 23:59:00"

In [25]:
val snap1 = ts(0)._2
val snap2 = ts(1)._2
val snap3 = ts(2)._2

snap1: Long = 8407173578228161952L
snap2: Long = 1040870493480983937L
snap3: Long = 5995907169292718909L

In [26]:
spark.sql(s"CALL lakekeeper.system.rollback_to_snapshot('spark_demo.taxis', $snap1)")

res26: DataFrame = [previous_snapshot_id: bigint, current_snapshot_id: bigint]

In [27]:
spark.table("spark_demo.taxis").show()

+---------+-------+-------------+-----------+------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|
+---------+-------+-------------+-----------+------------------+
+---------+-------+-------------+-----------+------------------+



In [28]:
spark.sql("SELECT * FROM spark_demo.taxis.snapshots ORDER BY committed_at").show()

+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|        committed_at|        snapshot_id|          parent_id|operation|       manifest_list|             summary|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+
|2026-07-02 23:34:...|8407173578228161952|               NULL|   append|s3://lakehouse/01...|{iceberg-version ...|
|2026-07-02 23:44:...|1040870493480983937|8407173578228161952|   append|s3://lakehouse/01...|{total-files-size...|
|2026-07-02 23:54:...|5995907169292718909|1040870493480983937|   append|s3://lakehouse/01...|{added-data-files...|
+--------------------+-------------------+-------------------+---------+--------------------+--------------------+



In [40]:
spark.sql(s"CALL lakekeeper.system.rollback_to_timestamp('spark_demo.taxis', TIMESTAMP '$ts2')")

res40: DataFrame = [previous_snapshot_id: bigint, current_snapshot_id: bigint]

In [41]:
spark.table("spark_demo.taxis").show()

+---------+-------+-------------+-----------+------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|
+---------+-------+-------------+-----------+------------------+
|        1|1000371|          1.8|      15.32|                 N|
|        2|1000372|          2.5|      22.15|                 N|
|        2|1000373|          0.9|       9.01|                 N|
|        1|1000374|          8.4|      42.13|                 Y|
+---------+-------+-------------+-----------+------------------+



In [31]:
spark.sql(s"CALL lakekeeper.system.set_current_snapshot('spark_demo.taxis', $snap3)")

res31: DataFrame = [previous_snapshot_id: bigint, current_snapshot_id: bigint]

In [32]:
spark.table("spark_demo.taxis").show()

+---------+-------+-------------+-----------+------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|
+---------+-------+-------------+-----------+------------------+
|        1|1000371|          1.8|      15.32|                 N|
|        2|1000372|          2.5|      22.15|                 N|
|        2|1000373|          0.9|       9.01|                 N|
|        3|2000371|          1.8|      15.32|                 N|
|        1|1000374|          8.4|      42.13|                 Y|
|        4|2000372|          2.5|      22.15|                 N|
|        4|2000373|          0.9|       9.01|                 N|
|        3|2000374|          8.4|      42.13|                 Y|
+---------+-------+-------------+-----------+------------------+



# Inspecting tables

In [33]:
spark.sql("SELECT * FROM spark_demo.taxis.history ORDER BY made_current_at").show(false)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-07-02 23:34:20.831|8407173578228161952|NULL               |true               |
|2026-07-02 23:44:23.692|1040870493480983937|8407173578228161952|true               |
|2026-07-02 23:54:26.021|5995907169292718909|1040870493480983937|true               |
|2026-07-02 23:54:30.003|8407173578228161952|NULL               |true               |
|2026-07-02 23:54:31.06 |5995907169292718909|1040870493480983937|true               |
+-----------------------+-------------------+-------------------+-------------------+



# Time travel

In [34]:
spark.sql(s"SELECT * FROM spark_demo.taxis TIMESTAMP AS OF '$ts2'").show()

+---------+-------+-------------+-----------+------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|
+---------+-------+-------------+-----------+------------------+
|        1|1000371|          1.8|      15.32|                 N|
|        2|1000372|          2.5|      22.15|                 N|
|        2|1000373|          0.9|       9.01|                 N|
|        1|1000374|          8.4|      42.13|                 Y|
+---------+-------+-------------+-----------+------------------+



In [35]:
spark.sql(s"SELECT * FROM spark_demo.taxis VERSION AS OF $snap1").show()

+---------+-------+-------------+-----------+------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|
+---------+-------+-------------+-----------+------------------+
+---------+-------+-------------+-----------+------------------+



In [36]:
spark.sql("SELECT * FROM spark_demo.taxis VERSION AS OF 'first_write'").show()

+---------+-------+-------------+-----------+------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|
+---------+-------+-------------+-----------+------------------+
|        1|1000371|          1.8|      15.32|                 N|
|        2|1000372|          2.5|      22.15|                 N|
|        2|1000373|          0.9|       9.01|                 N|
|        1|1000374|          8.4|      42.13|                 Y|
+---------+-------+-------------+-----------+------------------+



Metadata tables can also be inspected with time travel using the DataFrameReader API

In [37]:
val taxis = spark
            .read
            .format("iceberg")
            .option("snapshot-id", snap2)
            .load("spark_demo.taxis")

taxis.show()

+---------+-------+-------------+-----------+------------------+
|vendor_id|trip_id|trip_distance|fare_amount|store_and_fwd_flag|
+---------+-------+-------------+-----------+------------------+
|        1|1000371|          1.8|      15.32|                 N|
|        2|1000372|          2.5|      22.15|                 N|
|        2|1000373|          0.9|       9.01|                 N|
|        1|1000374|          8.4|      42.13|                 Y|
+---------+-------+-------------+-----------+------------------+



taxis: DataFrame = [vendor_id: bigint, trip_id: bigint ... 3 more fields]

# Files

In [38]:
spark.sql("""SELECT
    content,
    file_path,
    file_format,
    spec_id,
    record_count,
    file_size_in_bytes
FROM spark_demo.taxis.files""")
.show()

+-------+--------------------+-----------+-------+------------+------------------+
|content|           file_path|file_format|spec_id|record_count|file_size_in_bytes|
+-------+--------------------+-----------+-------+------------+------------------+
|      0|s3://lakehouse/01...|    PARQUET|      0|           1|              1553|
|      0|s3://lakehouse/01...|    PARQUET|      0|           1|              1550|
|      0|s3://lakehouse/01...|    PARQUET|      0|           1|              1553|
|      0|s3://lakehouse/01...|    PARQUET|      0|           1|              1553|
|      0|s3://lakehouse/01...|    PARQUET|      0|           1|              1553|
|      0|s3://lakehouse/01...|    PARQUET|      0|           1|              1548|
|      0|s3://lakehouse/01...|    PARQUET|      0|           1|              1552|
|      0|s3://lakehouse/01...|    PARQUET|      0|           1|              1553|
+-------+--------------------+-----------+-------+------------+------------------+



# Table Statistics

In [39]:
spark.sql("CALL lakekeeper.system.compute_table_stats('spark_demo.taxis')").show(false)

+---------------------------------------------------------------------------------------------------------------------------+
|statistics_file                                                                                                            |
+---------------------------------------------------------------------------------------------------------------------------+
|s3://lakehouse/019f248a-1c62-7c21-9b34-881b0477c529/metadata/5995907169292718909-116701b0-291b-42b5-94cf-8111dc3ab97d.stats|
+---------------------------------------------------------------------------------------------------------------------------+

